# Noise Channels and Recoverability

Notebook 23 asks:

**Are all disturbances equally removable?**

The answer is no.

Differential readout removes disturbances with the right symmetry.

- Common-mode noise cancels.
- Local noise remains as a precision floor.
- Signal-like disturbances survive because they have the same symmetry as the physical differential signal.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
FIGURES = ROOT / "figures"
DATA = ROOT / "data"

FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

rng = np.random.default_rng(42)


## Channel model

We use three disturbance channels:

1. **Common-mode noise**

\[
\phi_A = +s/2 + n_c
\]

\[
\phi_B = -s/2 + n_c
\]

This cancels in \(\phi_A - \phi_B\).

2. **Local noise**

\[
\phi_A = +s/2 + n_A
\]

\[
\phi_B = -s/2 + n_B
\]

This remains as \(n_A - n_B\).

3. **Signal-like disturbance**

\[
\phi_A = +s/2 + q/2
\]

\[
\phi_B = -s/2 - q/2
\]

This survives as \(q\), just like the true signal.


In [ ]:
n = 1000
t = np.linspace(0, 10, n)

signal = 0.3 * np.sin(2 * np.pi * 0.5 * t)

common_noise = 1.5 * rng.normal(size=n)
local_a = 0.25 * rng.normal(size=n)
local_b = 0.25 * rng.normal(size=n)
signal_like = 0.4 * np.sin(2 * np.pi * 0.9 * t + 0.5)


## Example channels

In [ ]:
phi_a_common = signal / 2 + common_noise
phi_b_common = -signal / 2 + common_noise
diff_common = phi_a_common - phi_b_common

phi_a_local = signal / 2 + local_a
phi_b_local = -signal / 2 + local_b
diff_local = phi_a_local - phi_b_local

phi_a_signal_like = signal / 2 + signal_like / 2
phi_b_signal_like = -signal / 2 - signal_like / 2
diff_signal_like = phi_a_signal_like - phi_b_signal_like


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(t, diff_common - signal, label="common-mode residual")
plt.plot(t, diff_local - signal, label="local residual", alpha=0.8)
plt.plot(t, diff_signal_like - signal, label="signal-like residual", alpha=0.8)

plt.xlabel("time")
plt.ylabel("residual after differential readout")

plt.title("Different noise channels leave different residuals")
plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "23_channel_examples.png",
    dpi=200
)

plt.show()


## Recoverability comparison

In [ ]:
rmse_common = np.sqrt(np.mean((diff_common - signal) ** 2))
rmse_local = np.sqrt(np.mean((diff_local - signal) ** 2))
rmse_signal_like = np.sqrt(np.mean((diff_signal_like - signal) ** 2))

recoverability = pd.DataFrame({
    "channel": [
        "common-mode",
        "local",
        "signal-like"
    ],
    "residual_rmse": [
        rmse_common,
        rmse_local,
        rmse_signal_like
    ],
    "recoverability": [
        "removed by subtraction",
        "sets precision floor",
        "survives differential readout"
    ]
})

recoverability


In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    recoverability["channel"],
    recoverability["residual_rmse"]
)

plt.ylabel("Residual RMSE")
plt.title("Recoverability depends on channel symmetry")

plt.tight_layout()

plt.savefig(
    FIGURES / "23_recoverability_bar.png",
    dpi=200
)

plt.show()


## Channel sweep

Now we sweep the amplitude of each channel and measure how much residual error remains after differential readout.


In [ ]:
amps = np.linspace(0, 2, 30)

rows = []

for amp in amps:

    # Common-mode channel
    common = amp * rng.normal(size=n)

    phi_a = signal / 2 + common
    phi_b = -signal / 2 + common
    phi_diff = phi_a - phi_b

    rows.append({
        "channel": "common-mode",
        "amplitude": amp,
        "residual_rmse": np.sqrt(np.mean((phi_diff - signal) ** 2))
    })

    # Local channel
    local_a = amp * rng.normal(size=n)
    local_b = amp * rng.normal(size=n)

    phi_a = signal / 2 + local_a
    phi_b = -signal / 2 + local_b
    phi_diff = phi_a - phi_b

    rows.append({
        "channel": "local",
        "amplitude": amp,
        "residual_rmse": np.sqrt(np.mean((phi_diff - signal) ** 2))
    })

    # Signal-like channel
    q = amp * np.sin(2 * np.pi * 0.9 * t + 0.5)

    phi_a = signal / 2 + q / 2
    phi_b = -signal / 2 - q / 2
    phi_diff = phi_a - phi_b

    rows.append({
        "channel": "signal-like",
        "amplitude": amp,
        "residual_rmse": np.sqrt(np.mean((phi_diff - signal) ** 2))
    })

channel_sweep = pd.DataFrame(rows)


In [ ]:
plt.figure(figsize=(8, 5))

for channel, group in channel_sweep.groupby("channel"):
    plt.plot(
        group["amplitude"],
        group["residual_rmse"],
        marker="o",
        label=channel
    )

plt.xlabel("Channel amplitude")
plt.ylabel("Residual RMSE")

plt.title("Differential readout removes only common-mode disturbances")
plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "23_channel_sweep.png",
    dpi=200
)

plt.show()


In [ ]:
recoverability.to_csv(
    DATA / "23_noise_channel_recoverability.csv",
    index=False
)

channel_sweep.to_csv(
    DATA / "23_noise_channel_sweep.csv",
    index=False
)

recoverability


# Lesson

Differential readout does not remove all noise.

It removes disturbances with common-mode symmetry.

Local disturbances remain as a precision floor.

Differential-mode disturbances survive because they look like signal.

Notebook 29 will introduce differential geometries and show how sensor layout determines which modes are rejected and which modes are measured.
